In [9]:
# Sensitivity analyses accompanying the main persistent postsurgical pain results:
# confounder adjustment, non-persistent-pain specificity, MNAR tipping-point
import os
import pandas as pd
import numpy as np
from helper_functions import *
from google.cloud import bigquery

directory = 'notebooks/data'

In [10]:
# The Workspace Bucket path and BigQuery dataset are read from a local file that
# is not included in this repository. Expected format, one entry per line:
#     bucket = ~/workspace/rw-migration-...
#     BQ_dataset = wb-...-....C....
config_path = os.path.expanduser('KimWhibley_config.txt')

config = {}
with open(config_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        key, value = line.split('=', 1)
        config[key.strip()] = value.strip()

bucket = config['bucket']
BQ_dataset = config['BQ_dataset']

In [11]:
job_query_config = bigquery.QueryJobConfig(
    default_dataset=BQ_dataset
)

client = bigquery.Client(
    default_query_job_config=job_query_config
)

In [12]:
combined_df = read_data_from_bucket("combined_df_main.csv",directory='notebooks/data', bucket = bucket)

In [13]:
# trim edges
combined_df = combined_df[
    (combined_df['days_before_onset'] >= 0) &
    (combined_df['days_before_onset'] <= 30)
]

## Secondary analyses - confounder adjustment & non-persistent-pain specificity

Robustness checks on the four pairs from the primary analysis: waketime
(gradual model), HR circadian amplitude (near-record), HR mesor (near-record), waketime
(near-record).

In [14]:
import statsmodels.formula.api as smf

# Persistent-pain-arm day-level dataframe

_csd = pd.to_datetime(combined_df["condition_start_date"]).dt.tz_localize(None)
_recon = (pd.to_datetime(combined_df["date"]).dt.tz_localize(None)
          + pd.to_timedelta(combined_df["days_before_onset"], unit="D"))
combined_df["condition_start_date"] = _csd.fillna(_recon)

print("Persistent-pain arm: {} people, {} records, {} person-day rows".format(
    combined_df["person_id"].nunique(),
    combined_df[["person_id", "condition_start_date"]].drop_duplicates().shape[0],
    len(combined_df)))

Persistent-pain arm: 98 people, 116 records, 3143 person-day rows


### Confounder-adjusted models
Charlson comorbidity index + six medication / psychiatric / sleep-disorder flags, each windowed
on every record's own `condition_start_date`.

In [15]:
# Confounder extraction
CHARLSON_LOOKBACK_DAYS = 365   # standard 1-year pre-index comorbidity lookback
ACUTE_WINDOW_DAYS = 30         # acute med window
CHRONIC_WINDOW_DAYS = (31, 395)  # chronic med window
CHRONIC_MIN_SPAN_DAYS = 90     # opioid fills must span >= this many days to count as "chronic"

CONFOUNDER_COLUMNS = [
    "charlson_index",
    "opioid_flag",
    "chronic_opioid_flag",
    "gabapentinoid_flag",
    "sedative_flag",
    "depression_anxiety_flag",
    "sleep_disorder_flag",
]
CONTINUOUS_CONFOUNDERS = ["charlson_index"]
BINARY_CONFOUNDERS = [c for c in CONFOUNDER_COLUMNS if c not in CONTINUOUS_CONFOUNDERS]

CONFOUNDER_WINDOWS = {
    "charlson_index":          "ICD-10-CM dx in the 365 d BEFORE the anchor date",
    "opioid_flag":             ">=1 opioid fill in the 30 d BEFORE the anchor date (acute window)",
    "chronic_opioid_flag":     "opioid fills spanning >=90 d within 31-395 d BEFORE the anchor date",
    "gabapentinoid_flag":      ">=1 gabapentinoid fill in the 30 d BEFORE the anchor date",
    "sedative_flag":           ">=1 sedative/hypnotic fill in the 30 d BEFORE the anchor date",
    "depression_anxiety_flag": "any depression/anxiety dx ANY TIME strictly before the anchor date",
    "sleep_disorder_flag":     "any sleep-disorder dx ANY TIME strictly before the anchor date",
}

# Charlson (Quan H et al., Med Care 2005; ICD-10-CM algorithm, original 1987 weights)
CHARLSON_ICD10_CODES = {
    "Myocardial infarction": ["I21", "I22", "I25.2"],
    "Congestive heart failure": ["I09.9", "I11.0", "I13.0", "I13.2", "I25.5", "I42.0",
                                 "I42.5-I42.9", "I43", "I50", "P29.0"],
    "Peripheral vascular disease": ["I70", "I71", "I73.1", "I73.8", "I73.9", "I77.1",
                                    "I79.0", "I79.2", "K55.1", "K55.8", "K55.9", "Z95.8", "Z95.9"],
    "Cerebrovascular disease": ["G45", "G46", "H34.0", "I60-I69"],
    "Dementia": ["F00-F03", "F05.1", "G30", "G31.1"],
    "Chronic pulmonary disease": ["J40-J47", "J60-J67", "J68.4", "J70.1", "J70.3"],
    "Rheumatic disease": ["M05", "M06", "M31.5", "M32-M34", "M35.1", "M35.3", "M36.0"],
    "Peptic ulcer disease": ["K25-K28"],
    "Mild liver disease": ["B18", "K70.0-K70.3", "K70.9", "K71.3-K71.5", "K71.7", "K73",
                           "K74", "K76.0", "K76.2-K76.4", "K76.8", "K76.9", "Z94.4"],
    "Diabetes without chronic complication": [
        "E10.0", "E10.1", "E10.6", "E10.8", "E10.9", "E11.0", "E11.1", "E11.6", "E11.8", "E11.9",
        "E12.0", "E12.1", "E12.6", "E12.8", "E12.9", "E13.0", "E13.1", "E13.6", "E13.8", "E13.9",
        "E14.0", "E14.1", "E14.6", "E14.8", "E14.9",
    ],
    "Diabetes with chronic complication": [
        "E10.2-E10.5", "E10.7", "E11.2-E11.5", "E11.7", "E12.2-E12.5", "E12.7",
        "E13.2-E13.5", "E13.7", "E14.2-E14.5", "E14.7",
    ],
    "Hemiplegia or paraplegia": ["G04.1", "G11.4", "G80.1", "G80.2", "G81", "G82", "G83.0-G83.4", "G83.9"],
    "Renal disease": ["I12.0", "I13.1", "N03.2-N03.7", "N05.2-N05.7", "N18", "N19", "N25.0",
                      "Z49.0-Z49.2", "Z94.0", "Z99.2"],
    "Malignancy": ["C00-C26", "C30-C34", "C37-C41", "C43", "C45-C58", "C60-C76", "C81-C85", "C88", "C90-C97"],
    "Moderate or severe liver disease": ["I85.0", "I85.9", "I86.4", "I98.2", "K70.4", "K71.1",
                                         "K72.1", "K72.9", "K76.5", "K76.6", "K76.7"],
    "Metastatic solid tumor": ["C77-C80"],
    "AIDS/HIV": ["B20-B22", "B24"],
}
CHARLSON_WEIGHTS = {
    "Myocardial infarction": 1, "Congestive heart failure": 1, "Peripheral vascular disease": 1,
    "Cerebrovascular disease": 1, "Dementia": 1, "Chronic pulmonary disease": 1, "Rheumatic disease": 1,
    "Peptic ulcer disease": 1, "Mild liver disease": 1, "Diabetes without chronic complication": 1,
    "Diabetes with chronic complication": 2, "Hemiplegia or paraplegia": 2, "Renal disease": 2,
    "Malignancy": 2, "Moderate or severe liver disease": 3, "Metastatic solid tumor": 6, "AIDS/HIV": 6,
}

# Drug ingredient names
OPIOID_INGREDIENT_NAMES = [
    "oxycodone", "hydrocodone", "morphine", "fentanyl", "hydromorphone",
    "codeine", "tramadol", "methadone", "buprenorphine", "oxymorphone", "tapentadol",
]
GABAPENTINOID_INGREDIENT_NAMES = ["gabapentin", "pregabalin"]
SEDATIVE_INGREDIENT_NAMES = [
    "zolpidem", "eszopiclone", "zaleplon", "trazodone",
    "diazepam", "lorazepam", "alprazolam", "clonazepam", "temazepam",
]

# SNOMED LIKE-patterns for the two unbounded-lookback history flags
DEPRESSION_ANXIETY_NAME_PATTERNS = ["%depress%", "%anxiety%", "%anxious%"]
SLEEP_DISORDER_NAME_PATTERNS = [
    "%insomnia%", "%sleep apnea%", "%sleep disorder%",
    "%hypersomnia%", "%narcolepsy%", "%restless legs%",
]

# Helpers
def _sql_id_list(ids):
    """Render an int iterable as a SQL ``IN`` list that is valid even for a
    single element (``(123)`` rather than ``(123,)``)."""
    ids = [int(x) for x in ids]
    if len(ids) == 1:
        return f"({ids[0]})"
    return "(" + ", ".join(str(x) for x in ids) + ")"


def _charlson_sql_condition(codes):
    """OR-clause matching any ICD-10-CM code root/range against ``c.concept_code``.
    A ``lo-hi`` entry matches the shared-length prefix range; anything else is a
    ``LIKE`` prefix."""
    clauses = []
    for code in codes:
        if "-" in code:
            lo, hi = code.split("-")
            n = min(len(lo), len(hi))
            clauses.append(f"SUBSTR(c.concept_code, 1, {n}) BETWEEN '{lo[:n]}' AND '{hi[:n]}'")
        else:
            clauses.append(f"c.concept_code LIKE '{code}%'")
    return "(" + " OR ".join(clauses) + ")"


def _normalize_pairs(pairs, id_col, date_col):
    """Return a clean 2-column frame ``[person_id, reference_date]`` (unique rows,
    tz-naive datetime64) plus the count of rows dropped for an unparseable date."""
    out = pairs[[id_col, date_col]].copy()
    out.columns = ["person_id", "reference_date"]
    out["person_id"] = out["person_id"].astype("int64")
    out["reference_date"] = pd.to_datetime(
        out["reference_date"], utc=True, errors="coerce"
    ).dt.tz_localize(None)
    n_bad = int(out["reference_date"].isna().sum())
    out = out.dropna(subset=["reference_date"]).drop_duplicates().reset_index(drop=True)
    return out, n_bad


def assert_valid_anchor_dates(pairs, valid_start=None, valid_end=None,
                              id_col="person_id", date_col="reference_date"):
    """
    ``valid_start`` / ``valid_end`` are either scalars or Series aligned to
    ``pairs.index`` (e.g. procedure_date + 90 d, and last observed date).
    """
    dt = pd.to_datetime(pairs[date_col], errors="coerce")
    n_null = int(dt.isna().sum())
    if n_null:
        raise ValueError(
            f"assert_valid_anchor_dates: {n_null} of {len(pairs)} anchor dates are "
            f"null/unparseable. Refusing to run confounder extraction with missing "
            f"anchors (would silently zero-fill every window)."
        )
    problems = []
    if valid_start is not None:
        vs = pd.to_datetime(pd.Series(valid_start, index=pairs.index), errors="coerce")
        bad = dt.values < vs.values
        if bad.any():
            problems.append(f"{int(bad.sum())} anchor(s) before the allowed window start")
    if valid_end is not None:
        ve = pd.to_datetime(pd.Series(valid_end, index=pairs.index), errors="coerce")
        bad = dt.values > ve.values
        if bad.any():
            problems.append(f"{int(bad.sum())} anchor(s) after the allowed window end")
    if problems:
        raise ValueError("assert_valid_anchor_dates: " + "; ".join(problems)
                         + ". Fix the reference-date draw before extracting confounders.")
    return True


# Charlson
def _charlson_index(client, pairs):
    person_ids = pairs["person_id"].unique()
    id_list = _sql_id_list(person_ids)

    union_parts = []
    for category, codes in CHARLSON_ICD10_CODES.items():
        cond = _charlson_sql_condition(codes)
        union_parts.append(f"""
            SELECT co.person_id, co.condition_start_date AS dx_date, '{category}' AS charlson_category
            FROM `condition_occurrence` co
            JOIN `concept` c ON co.condition_source_concept_id = c.concept_id
            WHERE c.vocabulary_id = 'ICD10CM' AND {cond}
              AND co.person_id IN {id_list}
        """)
    charlson_sql = "\nUNION ALL\n".join(union_parts)

    raw = client.query(charlson_sql).result().to_dataframe()
    raw["dx_date"] = pd.to_datetime(raw["dx_date"]).dt.tz_localize(None)

    merged = raw.merge(pairs, on="person_id", how="inner")
    merged["days_before_index"] = (merged["reference_date"] - merged["dx_date"]).dt.days
    in_window = merged[(merged["days_before_index"] >= 0)
                       & (merged["days_before_index"] <= CHARLSON_LOOKBACK_DAYS)]

    flags = (
        in_window
        .drop_duplicates(subset=["person_id", "reference_date", "charlson_category"])
        .assign(present=1)
        .pivot(index=["person_id", "reference_date"], columns="charlson_category", values="present")
        .reindex(columns=list(CHARLSON_WEIGHTS.keys()))
        .fillna(0).astype(int).reset_index()
    )

    # Standard Charlson mutual-exclusivity overrides - keep only the more severe form
    flags.loc[flags["Diabetes with chronic complication"] == 1,
              "Diabetes without chronic complication"] = 0
    flags.loc[flags["Moderate or severe liver disease"] == 1, "Mild liver disease"] = 0
    flags.loc[flags["Metastatic solid tumor"] == 1, "Malignancy"] = 0

    flags["charlson_index"] = sum(
        flags[cat] * weight for cat, weight in CHARLSON_WEIGHTS.items()
    )

    out = pairs.merge(flags[["person_id", "reference_date", "charlson_index"]],
                      on=["person_id", "reference_date"], how="left")
    out["charlson_index"] = out["charlson_index"].fillna(0).astype(int)
    return out[["person_id", "reference_date", "charlson_index"]]


# Drug-exposure flags
def _find_ingredient_concepts(client, ingredient_names):
    names_list = ", ".join(f"'{n.lower()}'" for n in ingredient_names)
    sql = f"""
        SELECT concept_id, concept_name
        FROM `concept`
        WHERE LOWER(concept_name) IN ({names_list})
          AND vocabulary_id = 'RxNorm'
          AND concept_class_id = 'Ingredient'
        ORDER BY concept_name;
    """
    return client.query(sql).result().to_dataframe()


def _flag_drug_exposure(client, pairs, ingredient_concept_ids, window_start_col,
                        window_end_col, flag_name, min_span_days=None):
    """For each (person_id, reference_date) row, flag a drug_exposure whose start
    date falls in ``[pairs[window_start_col], pairs[window_end_col]]``, rolled up to
    the given RxNorm Ingredient concept_ids via ``concept_ancestor``.
    """
    p = pairs.copy()
    if not ingredient_concept_ids:
        p[flag_name] = np.nan
        return p[["person_id", "reference_date", flag_name]]

    ingredient_ids = _sql_id_list(ingredient_concept_ids)
    person_ids = _sql_id_list(p["person_id"].unique())

    drug_sql = f"""
        SELECT DISTINCT de.person_id, de.drug_exposure_start_date
        FROM `drug_exposure` de
        JOIN `concept_ancestor` ca ON de.drug_concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id IN {ingredient_ids}
          AND de.person_id IN {person_ids}
        ORDER BY de.person_id, de.drug_exposure_start_date ASC;
    """
    drug_df = client.query(drug_sql).result().to_dataframe()
    drug_df["drug_exposure_start_date"] = (
        pd.to_datetime(drug_df["drug_exposure_start_date"]).dt.tz_localize(None)
    )

    merged = drug_df.merge(
        p[["person_id", "reference_date", window_start_col, window_end_col]],
        on="person_id", how="inner",
    )
    in_window = merged[
        (merged["drug_exposure_start_date"] >= merged[window_start_col])
        & (merged["drug_exposure_start_date"] <= merged[window_end_col])
    ]

    if min_span_days is None:
        flagged = in_window[["person_id", "reference_date"]].drop_duplicates()
    else:
        span = (
            in_window.groupby(["person_id", "reference_date"])["drug_exposure_start_date"]
            .agg(lambda s: (s.max() - s.min()).days)
            .reset_index(name="span_days")
        )
        flagged = span[span["span_days"] >= min_span_days][["person_id", "reference_date"]]

    p = p.merge(flagged.assign(**{flag_name: 1}),
                on=["person_id", "reference_date"], how="left")
    p[flag_name] = p[flag_name].fillna(0).astype(int)
    return p[["person_id", "reference_date", flag_name]]


# Condition-history flags (unbounded lookback)
def _discover_condition_ancestors(client, name_patterns):
    like_clause = " OR ".join(f"LOWER(c.concept_name) LIKE '{p}'" for p in name_patterns)
    sql = f"""
        SELECT c.concept_id, c.concept_name,
               COUNT(DISTINCT co.person_id) AS n_people
        FROM `concept` c
        LEFT JOIN `condition_occurrence` co ON co.condition_concept_id = c.concept_id
        WHERE c.domain_id = 'Condition'
          AND c.vocabulary_id = 'SNOMED'
          AND c.standard_concept = 'S'
          AND ({like_clause})
        GROUP BY c.concept_id, c.concept_name
        ORDER BY n_people DESC
    """
    return client.query(sql).result().to_dataframe()


def _flag_condition_history(client, pairs, ancestor_concept_ids, flag_name):
    """Flag any condition_occurrence (rolled up via ``concept_ancestor`` to the
    given broad SNOMED ancestors) with a start date STRICTLY before the anchor
    date -- unbounded lookback, same-day coding excluded."""
    p = pairs.copy()
    if not ancestor_concept_ids:
        p[flag_name] = np.nan
        return p[["person_id", "reference_date", flag_name]]

    ancestor_ids = _sql_id_list(ancestor_concept_ids)
    person_ids = _sql_id_list(p["person_id"].unique())

    cond_sql = f"""
        SELECT DISTINCT co.person_id, co.condition_start_date AS dx_date
        FROM `condition_occurrence` co
        JOIN `concept_ancestor` ca ON co.condition_concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id IN {ancestor_ids}
          AND co.person_id IN {person_ids}
        ORDER BY co.person_id, co.condition_start_date ASC;
    """
    cond_df = client.query(cond_sql).result().to_dataframe()
    cond_df["dx_date"] = pd.to_datetime(cond_df["dx_date"]).dt.tz_localize(None)

    merged = cond_df.merge(p[["person_id", "reference_date"]], on="person_id", how="inner")
    in_window = merged[merged["dx_date"] <= (merged["reference_date"] - pd.Timedelta(days=1))]
    flagged = in_window[["person_id", "reference_date"]].drop_duplicates()

    p = p.merge(flagged.assign(**{flag_name: 1}),
                on=["person_id", "reference_date"], how="left")
    p[flag_name] = p[flag_name].fillna(0).astype(int)
    return p[["person_id", "reference_date", flag_name]]


def extract_confounders(client, pairs, id_col="person_id", date_col="reference_date",
                        validate_dates=True):
    """Pull all seven confounders for an arbitrary set of (person, anchor-date) pairs.
    """
    if validate_dates:
        assert_valid_anchor_dates(pairs, id_col=id_col, date_col=date_col)

    base, n_bad = _normalize_pairs(pairs, id_col, date_col)
    if n_bad:
        print(f"WARNING: dropped {n_bad} row(s) with an unparseable {date_col}.")
    if base.empty:
        raise ValueError("extract_confounders: no usable (person_id, date) pairs after cleaning.")


    print(f"extract_confounders: {len(base)} unique (person_id, anchor-date) pairs, "
          f"{base['person_id'].nunique()} people.\n")

    # Window columns off the anchor date
    base = base.copy()
    base["acute_window_start"] = base["reference_date"] - pd.Timedelta(days=ACUTE_WINDOW_DAYS)
    base["acute_window_end"] = base["reference_date"]
    base["chronic_window_start"] = base["reference_date"] - pd.Timedelta(days=CHRONIC_WINDOW_DAYS[1])
    base["chronic_window_end"] = base["reference_date"] - pd.Timedelta(days=CHRONIC_WINDOW_DAYS[0])

    key = ["person_id", "reference_date"]
    out = base[key].copy()

    out = out.merge(_charlson_index(client, base[key]), on=key, how="left")

    # Drug/intervention flags
    opioid_ids = _find_ingredient_concepts(client, OPIOID_INGREDIENT_NAMES)["concept_id"].tolist()
    gaba_ids = _find_ingredient_concepts(client, GABAPENTINOID_INGREDIENT_NAMES)["concept_id"].tolist()
    sed_ids = _find_ingredient_concepts(client, SEDATIVE_INGREDIENT_NAMES)["concept_id"].tolist()

    out = out.merge(_flag_drug_exposure(client, base, opioid_ids,
                                        "acute_window_start", "acute_window_end", "opioid_flag"),
                    on=key, how="left")
    out = out.merge(_flag_drug_exposure(client, base, opioid_ids,
                                        "chronic_window_start", "chronic_window_end",
                                        "chronic_opioid_flag", min_span_days=CHRONIC_MIN_SPAN_DAYS),
                    on=key, how="left")
    out = out.merge(_flag_drug_exposure(client, base, gaba_ids,
                                        "acute_window_start", "acute_window_end", "gabapentinoid_flag"),
                    on=key, how="left")
    out = out.merge(_flag_drug_exposure(client, base, sed_ids,
                                        "acute_window_start", "acute_window_end", "sedative_flag"),
                    on=key, how="left")

    # Depression/anxiety + sleep-disorder history
    dep_ids = _discover_condition_ancestors(client, DEPRESSION_ANXIETY_NAME_PATTERNS)["concept_id"].tolist()
    slp_ids = _discover_condition_ancestors(client, SLEEP_DISORDER_NAME_PATTERNS)["concept_id"].tolist()

    out = out.merge(_flag_condition_history(client, base[key], dep_ids, "depression_anxiety_flag"),
                    on=key, how="left")
    out = out.merge(_flag_condition_history(client, base[key], slp_ids, "sleep_disorder_flag"),
                    on=key, how="left")

    out = out[key + CONFOUNDER_COLUMNS]

    return out

In [16]:
# one anchor per (person, persistent-pain record)
_pairs = combined_df[["person_id", "condition_start_date"]].drop_duplicates().copy()
_pairs["reference_date"] = pd.to_datetime(
    _pairs["condition_start_date"], utc=True, errors="coerce").dt.tz_localize(None)
assert_valid_anchor_dates(_pairs, id_col="person_id", date_col="reference_date")

confounder_df = extract_confounders(client, _pairs[["person_id", "reference_date"]])

combined_df = combined_df.drop(columns=[c for c in CONFOUNDER_COLUMNS if c in combined_df.columns])
combined_df["_k"] = pd.to_datetime(
    combined_df["condition_start_date"], utc=True, errors="coerce").dt.tz_localize(None)
combined_df = combined_df.merge(
    confounder_df.rename(columns={"reference_date": "_k"}),
    on=["person_id", "_k"], how="left").drop(columns="_k")
combined_df["charlson_index"] = combined_df["charlson_index"].fillna(0).astype(int)

# keep a confounder only if it is present and varies in this cohort
EXTRA_COVARIATES = [
    c for c in CONFOUNDER_COLUMNS
    if c in combined_df.columns and combined_df[c].notna().any()
    and combined_df[c].nunique(dropna=True) > 1
]

extract_confounders: 116 unique (person_id, anchor-date) pairs, 98 people.



In [17]:
# minimal mixed-model fitters (random person_id intercept, REML)
# gradual model: days_rel_onset ; near-record model: days_before_onset (days 0-2 vs. rest)
def _fit_day(d, y, covs=()):
    cols = [y, "days_rel_onset", "person_id", "age", "sex_at_birth_source_value", *covs]
    d = d[cols].replace([np.inf, -np.inf], np.nan).dropna()
    f = "{} ~ days_rel_onset + age + C(sex_at_birth_source_value)".format(y) + "".join(" + " + c for c in covs)
    return smf.mixedlm(f, d, groups=d["person_id"]).fit(reml=True)

def _fit_near(d, y, covs=()):
    cols = [y, "days_before_onset", "person_id", "age", "sex_at_birth_source_value", *covs]
    d = d[cols].replace([np.inf, -np.inf], np.nan).dropna()
    d = d[d["days_before_onset"] <= 30].copy()
    d["near_onset"] = d["days_before_onset"].isin([0, 1, 2]).astype(int)
    f = "{} ~ near_onset + age + C(sex_at_birth_source_value)".format(y) + "".join(" + " + c for c in covs)
    return smf.mixedlm(f, d, groups=d["person_id"]).fit(reml=True)

_PRIMARY_PAIRS = [
    ("waketime_hours", "days_rel_onset", _fit_day,  "Waketime (gradual)"),
    ("mean_hr_amp",    "near_onset",     _fit_near, "HR amplitude (near-record)"),
    ("mean_hr_mesor",  "near_onset",     _fit_near, "HR MESOR (near-record)"),
    ("waketime_hours", "near_onset",     _fit_near, "Waketime (near-record)"),
]

_rows = []
for _y, _term, _fn, _label in _PRIMARY_PAIRS:
    _rc = _fn(combined_df, _y)                     # baseline: age + sex only
    _ra = _fn(combined_df, _y, EXTRA_COVARIATES)   # adjusted: + confounder block
    _rows.append({
        "pair": _label,
        "baseline b (SE)":    "{:.4f} ({:.4f})".format(_rc.fe_params[_term], _rc.bse[_term]),
        "baseline p":         "{:.2g}".format(_rc.pvalues[_term]),
        "adjusted b (SE)": "{:.4f} ({:.4f})".format(_ra.fe_params[_term], _ra.bse[_term]),
        "adjusted p":      "{:.2g}".format(_ra.pvalues[_term]),
        "N obs / groups":  "{} / {}".format(int(_rc.nobs), len(_rc.random_effects)),
    })
confounder_primary_table = pd.DataFrame(_rows)
print("baseline vs. confounder-adjusted, four primary pairs:\n")
print(confounder_primary_table.to_string(index=False))

baseline vs. confounder-adjusted, four primary pairs:

                      pair  baseline b (SE) baseline p  adjusted b (SE) adjusted p N obs / groups
        Waketime (gradual) -0.0124 (0.0059)      0.035 -0.0114 (0.0059)      0.053      2159 / 90
HR amplitude (near-record)  4.2360 (0.8416)    4.8e-07  4.2299 (0.8411)    4.9e-07      2671 / 92
    HR MESOR (near-record)  4.2744 (0.8629)    7.3e-07  4.2944 (0.8627)    6.4e-07      2671 / 92
    Waketime (near-record) -0.6214 (0.1847)    0.00077 -0.6088 (0.1841)    0.00094      2159 / 90


### Non-persistent-pain specificity
Size-matched comparison arm: 98 surgical patients never diagnosed with persistent-pain (or any
coded variant, ever), each with >= 1 yr of confirmed subsequent persistent-pain-free follow-up,
anchored on a real non-pain clinical encounter. Extraction is reloaded from
`nonPPSP_realevent_*_main.csv`

Model per pair: `metric ~ time * C(visit_type) + age + C(sex) + (1 | person)`, persistent-pain
arm as reference; the `time x visit_type` interaction is the specificity test. Gradual model
uses `days_rel_onset`, near-record uses `near_record`.

In [18]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# load non-persistent-pain comparison arm
_ns = read_data_from_bucket("nonppsp_realevent_sleep_main.csv",    directory=directory, bucket=bucket)
_nc = read_data_from_bucket("nonppsp_realevent_circadian_main.csv", directory=directory, bucket=bucket)
_nm = read_data_from_bucket("nonppsp_realevent_meta_main.csv",      directory=directory, bucket=bucket)
_ns["datetime"] = pd.to_datetime(_ns["datetime"])
_nc["date"]     = pd.to_datetime(_nc["date"])

_n = _ns.merge(_nc, left_on=["person_id", "datetime"], right_on=["person_id", "date"], how="outer")
_n["date"] = _n["datetime"].combine_first(_n["date"])
_n = _n.drop(columns=["datetime"])
if "days_before_onset_x" in _n.columns:
    _n["days_before_onset"] = _n["days_before_onset_x"].combine_first(_n["days_before_onset_y"])
    _n = _n.drop(columns=["days_before_onset_x", "days_before_onset_y"])
_n = _n[(_n["days_before_onset"] >= 0) & (_n["days_before_onset"] <= 30)]
_n["days_rel_onset"] = -_n["days_before_onset"]
_n = _n.merge(_nm[["person_id", "age", "sex_at_birth_source_value"]].drop_duplicates("person_id"),
              on="person_id", how="left")

def _make_long(d, label):
    o = d.copy()
    o["visit_type"] = label
    o["near_record"] = o["days_before_onset"].isin([0, 1, 2]).astype(int)
    keep = ["person_id", "days_before_onset", "days_rel_onset", "near_record", "visit_type",
            "age", "sex_at_birth_source_value", "waketime_hours", "mean_hr_mesor", "mean_hr_amp"]
    return o[[c for c in keep if c in o.columns]]

nonppsp_long = pd.concat([_make_long(combined_df, "pain_related"),
                          _make_long(_n, "unrelated_nonppsp")], ignore_index=True)
print(nonppsp_long.groupby("visit_type")["person_id"].nunique().to_string())


def _fit_mlm(f, d):
    """Fit the mixed model. If the default optimizer raises a ConvergenceWarning, retry with
    lbfgs then cg. Returns (result, optimizer_used); the failure is reported in the table's
    'optimizer' column, not silenced."""
    for method in (None, ["lbfgs"], ["cg"]):
        with warnings.catch_warnings(record=True) as _w:
            warnings.simplefilter("always", ConvergenceWarning)
            r = smf.mixedlm(f, d, groups=d["person_id"]).fit(
                reml=True, **({} if method is None else {"method": method}))
        if not any(issubclass(x.category, ConvergenceWarning) for x in _w):
            return r, ("default" if method is None else method[0])
    return r, "none converged"


def _specificity(y, model):
    time = "days_rel_onset" if model == "day" else "near_record"
    d = nonppsp_long[[y, time, "person_id", "age", "sex_at_birth_source_value", "visit_type"]]
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    fits, opts = {}, set()
    for ref in ("pain_related", "unrelated_nonppsp"):
        f = ("{} ~ {} * C(visit_type, Treatment(reference='{}')) "
             "+ age + C(sex_at_birth_source_value)").format(y, time, ref)
        fits[ref], _o = _fit_mlm(f, d)
        opts.add(_o)
    ppsp, nonp = fits["pain_related"], fits["unrelated_nonppsp"]
    inter = next(t for t in ppsp.fe_params.index if t.startswith(time) and ":" in t)
    fmt = lambda r, t: "{:.4f} ({:.4f}), p={:.2g}".format(r.fe_params[t], r.bse[t], r.pvalues[t])
    return {
        "PPSP arm b (SE), p":      fmt(ppsp, time),
        "non-PPSP arm b (SE), p":  fmt(nonp, time),
        "cohort x time b (SE), p": fmt(ppsp, inter),
        "N obs / groups": "{} / {}".format(int(ppsp.nobs), len(ppsp.random_effects)),
        "optimizer": "default" if opts == {"default"} else ", ".join(sorted(opts)),
    }

nonppsp_primary_table = pd.DataFrame([
    {"pair": "Waketime (gradual)",         **_specificity("waketime_hours", "day")},
    {"pair": "HR amplitude (near-record)", **_specificity("mean_hr_amp",    "near")},
    {"pair": "HR MESOR (near-record)",     **_specificity("mean_hr_mesor",  "near")},
    {"pair": "Waketime (near-record)",     **_specificity("waketime_hours", "near")},
])
print("\nPersistent-pain vs. non-persistent-pain, four primary pairs:\n")
print(nonppsp_primary_table.to_string(index=False))
_flag = nonppsp_primary_table[nonppsp_primary_table["optimizer"] != "default"]
if len(_flag):
    print("\nConvergence note: default optimizer failed for -> "
          + "; ".join("{} (retried: {})".format(r.pair, r.optimizer) for r in _flag.itertuples()))

visit_type
pain_related         98
unrelated_nonppsp    98

Persistent-pain vs. non-persistent-pain, four primary pairs:

                      pair         PPSP arm b (SE), p   non-PPSP arm b (SE), p     cohort x time b (SE), p N obs / groups optimizer
        Waketime (gradual)  -0.0125 (0.0056), p=0.027 -0.0036 (0.0059), p=0.54     0.0089 (0.0081), p=0.27     4105 / 180   default
HR amplitude (near-record) 4.2203 (0.9106), p=3.6e-06 -0.1482 (0.9026), p=0.87 -4.3685 (1.2822), p=0.00066     5132 / 190        cg
    HR MESOR (near-record) 4.2637 (0.9229), p=3.8e-06  0.5387 (0.9185), p=0.56  -3.7250 (1.3022), p=0.0042     5132 / 190   default
    Waketime (near-record) -0.6242 (0.1764), p=0.0004 -0.1822 (0.1730), p=0.29    0.4420 (0.2471), p=0.074     4105 / 180   default

Convergence note: default optimizer failed for -> HR amplitude (near-record) (retried: cg)


### Full fixed-effects tables — Supplementary Tables S1 / S2 / S3 / S4

Model terms in **rows**, the four primary metric × model pairs in **columns**; every cell is `Estimate (SE), p` (`p` as `<.001` or 3 dp), in the supplement layout.

- **Table S1** ← `confounder_fe_adjusted` — persistent-pain arm, `metric ~ time + age + sex + Charlson + 6 medication / psychiatric / sleep flags`.
- **Table S2** ← `specificity_fe_crude` — persistent-pain vs. non-persistent-pain, `metric ~ time * visit_type + age + sex`; the `time × visit_type` interaction is the specificity test.
- **Table S3** ← `specificity_fe_adjusted` — Table S2 + Charlson + the 5 remaining flags in **both** arms (person-level "ever" join; `sedative_flag` excluded a priori — near-collinear with the cohort main effect, see `revision-doc.md` §1.2d).
- **Table S4** ← `table_s4` — record-/patient-level covariate balance between the persistent-pain arm (116 records) and the non-persistent-pain real-event comparison arm (98 patients).

The comparison arm throughout S2–S4 is the **real-event never-PPSP cohort** (`nonppsp_realevent_*_main.csv`): every reference date is the visit start of a real, coded, non-pain clinical encounter. The final cell re-prints **S1–S5 together in supplement order** (S5 = the MNAR tipping-point table from the cell near the top of this notebook).


In [19]:
# FULL FIXED-EFFECTS TABLES  --  Supplementary Tables S1 / S2 / S3

from datetime import date as _fe_date

FE_PAIRS = [
    ("Waketime (gradual)",          "waketime_hours", "day"),
    ("HR amplitude (near-record)",  "mean_hr_amp",    "near"),
    ("HR mesor (near-record)",      "mean_hr_mesor",  "near"),
    ("Waketime (near-record)",      "waketime_hours", "near"),
]

_VT = "C(visit_type, Treatment(reference='pain_related'))[T.unrelated_nonppsp]"

_CONF_LABELS = [
    ("charlson_index",          "Charlson comorbidity index"),
    ("opioid_flag",             "Opioid exposure (acute, 30 d)"),
    ("chronic_opioid_flag",     "Chronic opioid (31–395 d)"),
    ("gabapentinoid_flag",      "Gabapentinoid (acute, 30 d)"),
    ("sedative_flag",           "Sedative / hypnotic (acute, 30 d)"),
    ("depression_anxiety_flag", "Depression / anxiety history"),
    ("sleep_disorder_flag",     "Sleep-disorder history"),
]

_FOOT = ["Marginal R²", "Conditional R²", "N obs / N groups"]

_TIME_LABEL  = "Time (days relative to record / near-record)"
_VT_LABEL    = "Visit type (alternate vs. persistent pain)"
_INTER_LABEL = "Interaction (time × visit type)"

_CONF_LEAD = ["Intercept", _TIME_LABEL, "Sex: Female", "Sex: Male", "Age"]
_SPEC_LEAD = ["Intercept", _VT_LABEL, "Sex: Female", "Sex: Male",
              _TIME_LABEL, _INTER_LABEL, "Age"]

_SPEC_ADJ_EXCLUDE = {"sedative_flag"}   # near-collinear with cohort


def _nakagawa_r2(res, d):
    fe = res.predict(exog=d)
    v_fix = float(np.var(fe))
    v_ran = float(res.cov_re.iloc[0, 0])
    v_res = float(res.scale)
    tot = v_fix + v_ran + v_res
    return v_fix / tot, (v_fix + v_ran) / tot


def _cell(res, term):
    if term is None or term not in res.fe_params.index:
        return ""
    b, se, p = res.fe_params[term], res.bse[term], res.pvalues[term]
    return f"{b:.3f} ({se:.3f}), " + ("<.001" if p < .001 else f"{p:.3f}")


def _fe_column(res, d, time_term, inter_term=None, conf_terms=()):
    col = {}
    col["Intercept"] = _cell(res, "Intercept")
    if inter_term is not None:
        col[_VT_LABEL] = _cell(res, _VT)
    for t in res.fe_params.index:
        if t.startswith("C(sex_at_birth_source_value)[T."):
            lvl = t.split("[T.", 1)[1].rstrip("]").replace("SexAtBirth_", "")
            col[f"Sex: {lvl}"] = _cell(res, t)
    col[_TIME_LABEL] = _cell(res, time_term)
    if inter_term is not None:
        col[_INTER_LABEL] = _cell(res, inter_term)
    col["Age"] = _cell(res, "age")
    for t, lab in _CONF_LABELS:
        if t in conf_terms:
            col[lab] = _cell(res, t)
    r2m, r2c = _nakagawa_r2(res, d)
    col["Marginal R²"]    = round(float(r2m), 3)
    col["Conditional R²"] = round(float(r2c), 3)
    col["N obs / N groups"]    = f"{int(res.nobs)} / {int(len(res.random_effects))}"
    return col


def _assemble(cols_by_pair, lead_rows):
    rows = list(lead_rows)
    for col in cols_by_pair.values():
        for k in col:
            if k not in rows and k not in _FOOT:
                rows.append(k)
    rows = rows + _FOOT
    return pd.DataFrame(
        {p: [cols_by_pair[p].get(r, "") for r in rows] for p in cols_by_pair},
        index=rows,
    )


# confounder analysis, persistent-pain arm
def _fit_conf(y, model, covs=()):
    if model == "day":
        time_term = "days_rel_onset"
        need = [y, time_term, "person_id", "age", "sex_at_birth_source_value", *covs]
        d = combined_df[need].replace([np.inf, -np.inf], np.nan).dropna().copy()
        f = f"{y} ~ days_rel_onset + age + C(sex_at_birth_source_value)"
    else:
        time_term = "near_onset"
        need = [y, "days_before_onset", "person_id", "age", "sex_at_birth_source_value", *covs]
        d = combined_df[need].replace([np.inf, -np.inf], np.nan).dropna()
        d = d[d["days_before_onset"] <= 30].copy()
        d["near_onset"] = d["days_before_onset"].isin([0, 1, 2]).astype(int)
        f = f"{y} ~ near_onset + age + C(sex_at_birth_source_value)"
    if covs:
        f += "".join(" + " + c for c in covs)
    res = smf.mixedlm(f, d, groups=d["person_id"]).fit(reml=True)
    return res, d, time_term


_crude_cols, _adj_cols = {}, {}
for _name, _y, _model in FE_PAIRS:
    _rc, _dc, _tc = _fit_conf(_y, _model)
    _ra, _da, _ta = _fit_conf(_y, _model, EXTRA_COVARIATES)
    _crude_cols[_name] = _fe_column(_rc, _dc, _tc)
    _adj_cols[_name]   = _fe_column(_ra, _da, _ta, conf_terms=EXTRA_COVARIATES)

confounder_fe_crude    = _assemble(_crude_cols, _CONF_LEAD)
confounder_fe_adjusted = _assemble(_adj_cols,   _CONF_LEAD)


# specificity, persistent-pain vs. non-persistent-pain
def _fit_spec(src, y, model, covs=()):
    time = "days_rel_onset" if model == "day" else "near_record"
    need = [y, time, "days_before_onset", "person_id", "age",
            "sex_at_birth_source_value", "visit_type", *covs]
    d = src[need].replace([np.inf, -np.inf], np.nan)
    if model != "day":
        d = d[d["days_before_onset"] <= 30]
    d = d.dropna().copy()
    f = (f"{y} ~ {time} * C(visit_type, Treatment(reference='pain_related')) "
         f"+ age + C(sex_at_birth_source_value)")
    if covs:
        f += " + " + " + ".join(covs)
    res, opt = _fit_mlm(f, d)
    return res, d, time, f"{time}:{_VT}", opt


_spec_crude_cols, _spec_conv = {}, []
for _name, _y, _model in FE_PAIRS:
    _r, _d, _tt, _it, _opt = _fit_spec(nonppsp_long, _y, _model)
    _spec_crude_cols[_name] = _fe_column(_r, _d, _tt, inter_term=_it)
    _spec_conv.append(f"  {_name:<28s} crude    -> optimizer: {_opt}")
specificity_fe_crude = _assemble(_spec_crude_cols, _SPEC_LEAD)


# specificity + confounder adjustment (confounders joined person-level to both arms)
_np_meta = read_data_from_bucket("nonppsp_realevent_meta_main.csv",
                                 directory=directory, bucket=bucket)
_np_pairs = _np_meta[["person_id", "reference_date"]].drop_duplicates().copy()
_np_pairs["reference_date"] = pd.to_datetime(_np_pairs["reference_date"])
if _np_pairs["reference_date"].dt.tz is not None:
    _np_pairs["reference_date"] = _np_pairs["reference_date"].dt.tz_localize(None)
assert_valid_anchor_dates(_np_pairs, id_col="person_id", date_col="reference_date")
print("extracting non-persistent-pain-arm confounders ...")
_np_conf = extract_confounders(client, _np_pairs)

_CONF_COLS = [c for c, _ in _CONF_LABELS]


def _person_conf(df):
    agg = {"charlson_index": "mean"}
    agg.update({c: "max" for c in _CONF_COLS if c != "charlson_index" and c in df.columns})
    return df.groupby("person_id").agg(agg).reset_index()


_pc_PPSP = _person_conf(combined_df[["person_id"] + [c for c in _CONF_COLS if c in combined_df.columns]])
_pc_np = _person_conf(_np_conf)

_spec_adj = ["charlson_index"]
for _c in _CONF_COLS:
    if _c == "charlson_index":
        continue
    if _c in _SPEC_ADJ_EXCLUDE:
        print(f"  specificity adjustment: dropped {_c} "
              f"(excluded a priori -- near-collinear with cohort; revision-doc.md 1.2d)")
        continue
    _p_pain = float(_pc_PPSP[_c].mean()) if _c in _pc_PPSP.columns else 0.0
    _p_np = float(_pc_np[_c].mean()) if _c in _pc_np.columns else 0.0
    if min(_p_pain, _p_np) >= 0.02:
        _spec_adj.append(_c)
    else:
        print(f"  specificity adjustment: dropped {_c} "
              f"(person-level prevalence {_p_pain:.1%} persistent-pain / {_p_np:.1%} non-persistent-pain)")

_pc_all = pd.concat([_pc_PPSP, _pc_np], ignore_index=True).drop_duplicates("person_id")
_long_adj = nonppsp_long.merge(_pc_all, on="person_id", how="left")
for _c in _spec_adj:
    _long_adj[_c] = _long_adj[_c].fillna(0)

_spec_adj_cols = {}
for _name, _y, _model in FE_PAIRS:
    _r, _d, _tt, _it, _opt = _fit_spec(_long_adj, _y, _model, _spec_adj)
    _spec_adj_cols[_name] = _fe_column(_r, _d, _tt, inter_term=_it, conf_terms=_spec_adj)
    _spec_conv.append(f"  {_name:<28s} adjusted -> optimizer: {_opt}")
specificity_fe_adjusted = _assemble(_spec_adj_cols, _SPEC_LEAD)


# print
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", None)


def _show(title, tbl):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)
    print(tbl.to_string())


_show("Table S1  --  confounder-adjusted mixed-effects models, persistent-pain arm "
      "(metric ~ time + age + sex + Charlson + 6 flags)",
      confounder_fe_adjusted)
_show("Table S2  --  cohort / visit-type specificity models, crude "
      "(metric ~ time * visit_type + age + sex)",
      specificity_fe_crude)
_show("Table S3  --  cohort / visit-type specificity models, confounder-adjusted "
      "(Table S2 + Charlson + flags in both arms)",
      specificity_fe_adjusted)
_show("(internal reference -- not a supplement table)  crude confounder models, persistent-pain arm",
      confounder_fe_crude)

print("\nOptimizer per specificity fit:")
print("\n".join(_spec_conv))

print("\n" + "=" * 100)
print("Compact crude-vs-adjusted comparison (persistent-pain arm, from the cell above):")
print("=" * 100)
print(confounder_primary_table.to_string(index=False))


extracting non-persistent-pain-arm confounders ...
extract_confounders: 98 unique (person_id, anchor-date) pairs, 98 people.

  specificity adjustment: dropped sedative_flag (excluded a priori -- near-collinear with cohort; revision-doc.md 1.2d)

Table S1  --  confounder-adjusted mixed-effects models, persistent-pain arm (metric ~ time + age + sex + Charlson + 6 flags)
                                                 Waketime (gradual) HR amplitude (near-record)  HR mesor (near-record) Waketime (near-record)
Intercept                                      8.097 (3.085), 0.009       6.867 (5.854), 0.241  85.965 (10.311), <.001   8.323 (3.093), 0.007
Time (days relative to record / near-record)  -0.011 (0.006), 0.053       4.230 (0.841), <.001    4.294 (0.863), <.001  -0.609 (0.184), <.001
Sex: Female                                    0.547 (2.753), 0.842       4.295 (5.207), 0.409    7.110 (9.219), 0.441   0.544 (2.763), 0.844
Sex: Male                                      0.426 (2.805)

In [20]:
# Supplementary Table S4 
# Persistent-pain arm : one row per clinical record (N = 116); confounder flags
#                       from `confounder_df`, age / sex from `combined_df`.
# Comparison arm      : one row per patient (N = 98); confounder flags from
#                       `_np_conf` (extracted above), age / sex from `_np_meta`.

def _pct_ci(x):
    x = pd.Series(x).dropna().astype(float).values
    n = len(x)
    p = x.mean() if n else np.nan
    se = np.sqrt(p * (1 - p) / n) if n else np.nan
    lo, hi = max(0.0, p - 1.96 * se), min(1.0, p + 1.96 * se)
    return f"{100 * p:.1f}% [{100 * lo:.1f}, {100 * hi:.1f}]"


def _mean_sd(x, dp=1):
    x = pd.Series(x).dropna().astype(float).values
    return f"{x.mean():.{dp}f} ({x.std(ddof=1):.{dp}f})"


def _is_female(s):
    return s.astype(str).str.contains("Female", case=False, na=False).astype(float)


# persistent-pain arm, record level (N = 116 clinical records)
_pp_dem = (combined_df[["person_id", "condition_start_date", "age", "sex_at_birth_source_value"]]
           .drop_duplicates(["person_id", "condition_start_date"]).copy())
_pp_dem["condition_start_date"] = pd.to_datetime(_pp_dem["condition_start_date"]).dt.tz_localize(None)
_cf_s4 = confounder_df.copy()
_cf_s4["reference_date"] = pd.to_datetime(_cf_s4["reference_date"]).dt.tz_localize(None)
_pp_s4 = _pp_dem.merge(_cf_s4, left_on=["person_id", "condition_start_date"],
                       right_on=["person_id", "reference_date"], how="inner")

# non-persistent-pain real-event comparison arm, patient level (N = 98)
_np_dem = _np_meta[["person_id", "age", "sex_at_birth_source_value"]].drop_duplicates("person_id")
_np_s4 = _np_conf.merge(_np_dem, on="person_id", how="inner")

print(f"Table S4 inputs: persistent-pain arm matched {len(_pp_s4)}/116 records; "
      f"comparison arm {len(_np_s4)}/98 patients")

_S4_ROWS = [
    ("Age, mean (SD)",                         lambda d: _mean_sd(d["age"], 1)),
    ("Female, % [95% CI]",                     lambda d: _pct_ci(_is_female(d["sex_at_birth_source_value"]))),
    ("Charlson index, mean (SD)",              lambda d: _mean_sd(d["charlson_index"], 2)),
    ("Acute opioid, % [95% CI]",               lambda d: _pct_ci(d["opioid_flag"])),
    ("Chronic opioid, % [95% CI]",             lambda d: _pct_ci(d["chronic_opioid_flag"])),
    ("Gabapentinoid, % [95% CI]",              lambda d: _pct_ci(d["gabapentinoid_flag"])),
    ("Sedative/hypnotic, % [95% CI]",          lambda d: _pct_ci(d["sedative_flag"])),
    ("Depression/anxiety history, % [95% CI]", lambda d: _pct_ci(d["depression_anxiety_flag"])),
    ("Sleep-disorder history, % [95% CI]",     lambda d: _pct_ci(d["sleep_disorder_flag"])),
]

_pp_label = f"Persistent pain arm ({len(_pp_s4)} rec / {_pp_s4['person_id'].nunique()} ppl)"
_np_label = f"Non-persistent-pain real-event arm ({len(_np_s4)})"
table_s4 = pd.DataFrame(
    {_pp_label: [fn(_pp_s4) for _, fn in _S4_ROWS],
     _np_label: [fn(_np_s4) for _, fn in _S4_ROWS]},
    index=[r for r, _ in _S4_ROWS],
)
table_s4.index.name = "Variable"

print("\n" + "=" * 100)
print("Table S4  --  covariate balance: persistent-pain arm vs. non-persistent-pain real-event arm")
print("=" * 100)
print(table_s4.to_string())
table_s4


Table S4 inputs: persistent-pain arm matched 116/116 records; comparison arm 98/98 patients

Table S4  --  covariate balance: persistent-pain arm vs. non-persistent-pain real-event arm
                                       Persistent pain arm (116 rec / 98 ppl) Non-persistent-pain real-event arm (98)
Variable                                                                                                             
Age, mean (SD)                                                    56.3 (13.7)                             51.2 (14.6)
Female, % [95% CI]                                         78.4% [71.0, 85.9]                      83.7% [76.4, 91.0]
Charlson index, mean (SD)                                         1.99 (2.07)                             0.68 (1.30)
Acute opioid, % [95% CI]                                   57.8% [48.8, 66.7]                         2.0% [0.0, 4.8]
Chronic opioid, % [95% CI]                                 21.6% [14.1, 29.0]                         3.1% 

,Persistent pain arm (116 rec / 98 ppl),Non-persistent-pain real-event arm (98)
Variable,,
"Age, mean (SD)",56.3 (13.7),51.2 (14.6)
"Female, % [95% CI]","78.4% [71.0, 85.9]","83.7% [76.4, 91.0]"
"Charlson index, mean (SD)",1.99 (2.07),0.68 (1.30)
"Acute opioid, % [95% CI]","57.8% [48.8, 66.7]","2.0% [0.0, 4.8]"
"Chronic opioid, % [95% CI]","21.6% [14.1, 29.0]","3.1% [0.0, 6.5]"
"Gabapentinoid, % [95% CI]","22.4% [14.8, 30.0]","2.0% [0.0, 4.8]"
"Sedative/hypnotic, % [95% CI]","12.9% [6.8, 19.0]","2.0% [0.0, 4.8]"
"Depression/anxiety history, % [95% CI]","63.8% [55.0, 72.5]","45.9% [36.1, 55.8]"
"Sleep-disorder history, % [95% CI]","61.2% [52.3, 70.1]","37.8% [28.2, 47.4]"


## Secondary analysis — MNAR sensitivity (Supplementary Table S5)

Tipping-point (delta-adjustment) sensitivity for the four primary metric × model pairs. Subjects whose wearable series stopped before the clinical-record day have that day's value multiply-imputed under MAR, the imputed encounter-day values are shifted by `delta`, and the model is refit; estimates are pooled across imputations with Rubin's rules. `delta` is swept over ± 1.5 SD of each metric.

The cell below consolidates the analysis into a single table in the **Table S5** layout — `delta` (SD units) in rows, the four pairs in columns, `Estimate (SE)` / `p-val` per cell. The MICE imputation is seeded (`MNAR_SEED`) so the table is reproducible; the originally written-up S5 came from an earlier unseeded run (`KimWhibley_Code.ipynb`), which the seeded `M = 20` result matches to within Monte-Carlo error.

In [21]:
# MNAR tipping-point sensitivity  --  Supplementary Table S5
# For each primary metric x model pair, subjects whose wearable data stopped
# before the clinical-record day (encounter_day = 0) get that day's value
# multiply-imputed (MICE, M = N_IMPUTATIONS) under MAR; the imputed encounter-day
# values are then shifted by `delta` (an MNAR departure) and the mixed model is
# refit. Slopes / near-onset coefficients are pooled across imputations with
# Rubin's rules. `delta` is swept over +/- 1.5 SD of each metric (7 points).

import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.imputation.mice import MICEData
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy import stats

N_IMPUTATIONS   = 20
MNAR_SEED       = 20260828
SD_MULTIPLES    = np.array([-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5])
ENCOUNTER_DAY   = 0
NEAR_ONSET_DAYS = (0, 1, 2)
FIT_WINDOW      = 30
_COVARS = ["age", "sex_at_birth_source_value"]


def _rubin_pool(estimates, variances):
    estimates = np.asarray(estimates, float)
    variances = np.asarray(variances, float)
    m = len(estimates)
    q_bar = estimates.mean()
    u_bar = variances.mean()
    b = estimates.var(ddof=1) if m > 1 else 0.0
    se_pooled = np.sqrt(u_bar + (1 + 1 / m) * b)
    if b > 0 and u_bar > 0:
        r = (1 + 1 / m) * b / u_bar
        dfree = (m - 1) * (1 + 1 / r) ** 2
    else:
        dfree = np.inf
    pval = 2 * (1 - stats.t.cdf(abs(q_bar / se_pooled), dfree))
    return q_bar, se_pooled, pval


def _flag_encounter_missingness(df, target, day_col, direction):
    """Add a was_missing=True encounter-day row for every subject whose observed
    series for `target` stopped before ENCOUNTER_DAY."""
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[[target, day_col, "person_id", *_COVARS]].dropna(subset=[day_col, "person_id"]).copy()
    has_t = df.dropna(subset=[target])
    has_encounter_row = set(has_t.loc[has_t[day_col] == ENCOUNTER_DAY, "person_id"])
    if direction == "increasing":          # day_col counts up; dropout truncates the max
        last_obs = has_t.groupby("person_id")[day_col].max()
        dropout = last_obs < ENCOUNTER_DAY
    else:                                   # 'decreasing': day_col counts down to 0
        last_obs = has_t.groupby("person_id")[day_col].min()
        dropout = last_obs > ENCOUNTER_DAY
    dropout_subjects = last_obs[dropout & ~last_obs.index.isin(has_encounter_row)].index.tolist()
    subj_meta = (df[["person_id", *_COVARS]].dropna()
                 .drop_duplicates("person_id").set_index("person_id"))
    missing_rows = pd.DataFrame({"person_id": dropout_subjects,
                                 day_col: ENCOUNTER_DAY, target: np.nan})
    missing_rows = missing_rows.merge(subj_meta, on="person_id", how="left")
    missing_rows["was_missing"] = True
    df["was_missing"] = False
    return pd.concat([df, missing_rows], ignore_index=True)


def _add_subj_mean(df, target):
    """Subject mean on observed days -- a stand-in for the random intercept inside
    MICEData's regression imputer."""
    sm = df.loc[~df["was_missing"]].groupby("person_id")[target].mean().rename("subj_mean")
    df = df.merge(sm, on="person_id", how="left")
    df["subj_mean"] = df["subj_mean"].fillna(df[target].mean())
    return df


def _impute_and_shift(df, target, day_col, delta):
    mice_df = df[[target, day_col, "age", "subj_mean"]].copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        np.random.seed(MNAR_SEED)          # deterministic MICE; common draws across the delta sweep
        md = MICEData(mice_df)
        md.set_imputer(target, formula=f"{day_col} + age + subj_mean")
        for _ in range(N_IMPUTATIONS):
            md.update_all()
            done = md.data.copy()
            done[target] = np.where(df["was_missing"].values,
                                    done[target] + delta, done[target])
            done["person_id"] = df["person_id"].values
            done["sex_at_birth_source_value"] = df["sex_at_birth_source_value"].values
            yield done


def _pooled_effect(flagged, target, day_col, delta, model_type):
    ests, vars = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for done in _impute_and_shift(flagged, target, day_col, delta):
            if model_type == "continuous":
                term, d = day_col, done
                f = f"{target} ~ {day_col} + age + C(sex_at_birth_source_value)"
            else:
                d = done[done[day_col] <= FIT_WINDOW].copy()
                d["near_onset"] = d[day_col].isin(NEAR_ONSET_DAYS).astype(int)
                term = "near_onset"
                f = f"{target} ~ near_onset + age + C(sex_at_birth_source_value)"
            res = smf.mixedlm(f, d, groups=d["person_id"]).fit(reml=True)
            ests.append(res.fe_params[term])
            vars.append(res.bse[term] ** 2)
    return _rubin_pool(ests, vars)


def _fmt_est(q, se):
    return f"{q:.3f} ({se:.3f})"


def _fmt_p(p):
    return "<.001" if p < 0.001 else round(float(p), 3)


# (label, target, model_type, day_col, direction)
_PAIRS = [
    ("Waketime (gradual)",         "waketime_hours", "continuous",  "days_rel_onset",    "increasing"),
    ("HR Amplitude (near-record)", "mean_hr_amp",    "categorical", "days_before_onset", "decreasing"),
    ("HR Mesor (near-record)",     "mean_hr_mesor",  "categorical", "days_before_onset", "decreasing"),
    ("Waketime (near-record)",     "waketime_hours", "categorical", "days_before_onset", "decreasing"),
]

_col_blocks, _sd_note = [], {}
for _label, _target, _mtype, _day_col, _dir in _PAIRS:
    _sd = combined_df[_target].std()
    _sd_note[_label] = _sd
    _flagged = _add_subj_mean(
        _flag_encounter_missingness(combined_df, _target, _day_col, _dir), _target)
    _est, _p = [], []
    for _k in SD_MULTIPLES:
        q, se, pv = _pooled_effect(_flagged, _target, _day_col, _k * _sd, _mtype)
        _est.append(_fmt_est(q, se))
        _p.append(_fmt_p(pv))
    _col_blocks.append(pd.DataFrame(
        {(_label, "Estimate (SE)"): _est, (_label, "p-val"): _p}, index=SD_MULTIPLES))

tipping_point_table = pd.concat(_col_blocks, axis=1)
tipping_point_table.columns = pd.MultiIndex.from_tuples(tipping_point_table.columns)
tipping_point_table.index = ["0 (MAR baseline)" if _k == 0 else f"{_k:g}" for _k in SD_MULTIPLES]
tipping_point_table.index.name = "Delta (SD units)"

print("Supplementary Table S5  --  MNAR tipping-point sensitivity of the primary models")
print(f"M = {N_IMPUTATIONS} imputations (seed {MNAR_SEED}); delta in SD units of each metric.")
print("1 SD in raw units:  " + "   ".join(f"{k} = {v:.3f}" for k, v in _sd_note.items()))
tipping_point_table


Supplementary Table S5  --  MNAR tipping-point sensitivity of the primary models
M = 20 imputations (seed 20260828); delta in SD units of each metric.
1 SD in raw units:  Waketime (gradual) = 3.093   HR Amplitude (near-record) = 13.076   HR Mesor (near-record) = 15.601   Waketime (near-record) = 3.093


Waketime (gradual)        HR Amplitude (near-record)        HR Mesor (near-record)        Waketime (near-record)       
                      Estimate (SE)  p-val              Estimate (SE)  p-val          Estimate (SE)  p-val          Estimate (SE)  p-val
Delta (SD units)                                                                                                                        
-1.5                 -0.025 (0.006)  <.001              1.395 (1.015)  0.171          1.083 (1.087)  0.321         -1.177 (0.176)  <.001
-1                   -0.021 (0.006)  <.001              2.194 (1.010)  0.031          2.036 (1.081)  0.062         -0.929 (0.175)  <.001
-0.5                 -0.016 (0.006)  0.008              2.992 (1.007)  0.003          2.989 (1.077)  0.006         -0.682 (0.174)  <.001
0 (MAR baseline)     -0.011 (0.006)  0.064              3.791 (1.005)  <.001          3.942 (1.075)  <.001         -0.434 (0.174)  0.014
0.5                  -0.006 (0.006)   0.29              4.590 (1.005)  <.001          4.896 (1.076)  <.001         -0.186 (0.174)  0.288
1                    -0.002 (0.006)  0.797              5.389 (1.007)  <.001          5.849 (1.078)  <.001          0.062 (0.175)  0.724
1.5                   0.003 (0.006)  0.594              6.187 (1.010)  <.001          6.802 (1.082)  <.001          0.310 (0.177)  0.082

In [22]:
# Supplementary Tables S1-S5
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", None)


def _show_supp(title, tbl):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)
    print(tbl.to_string())


_show_supp("Supplementary Table S1  --  confounder-adjusted mixed-effects models, persistent-pain arm "
           "(metric ~ time + age + sex + Charlson + 6 medication / psychiatric / sleep flags)",
           confounder_fe_adjusted)
_show_supp("Supplementary Table S2  --  cohort / visit-type specificity models, crude "
           "(metric ~ time * visit_type + age + sex; time x visit_type is the specificity test)",
           specificity_fe_crude)
_show_supp("Supplementary Table S3  --  cohort / visit-type specificity models, confounder-adjusted "
           "(Table S2 + Charlson + flags in both arms; comparison arm = real-event never-PPSP cohort)",
           specificity_fe_adjusted)
_show_supp("Supplementary Table S4  --  covariate balance between the persistent-pain arm and the "
           "non-persistent-pain real-event comparison arm",
           table_s4)
_show_supp("Supplementary Table S5  --  MNAR delta-adjustment (tipping-point) sensitivity of the "
           "four primary metric x model pairs",
           tipping_point_table)



Supplementary Table S1  --  confounder-adjusted mixed-effects models, persistent-pain arm (metric ~ time + age + sex + Charlson + 6 medication / psychiatric / sleep flags)
                                                 Waketime (gradual) HR amplitude (near-record)  HR mesor (near-record) Waketime (near-record)
Intercept                                      8.097 (3.085), 0.009       6.867 (5.854), 0.241  85.965 (10.311), <.001   8.323 (3.093), 0.007
Time (days relative to record / near-record)  -0.011 (0.006), 0.053       4.230 (0.841), <.001    4.294 (0.863), <.001  -0.609 (0.184), <.001
Sex: Female                                    0.547 (2.753), 0.842       4.295 (5.207), 0.409    7.110 (9.219), 0.441   0.544 (2.763), 0.844
Sex: Male                                      0.426 (2.805), 0.879       2.894 (5.296), 0.585    2.367 (9.364), 0.800   0.439 (2.814), 0.876
Age                                           -0.025 (0.023), 0.282      -0.046 (0.045), 0.298   -0.279 (0.076), <.00